In [1]:
import os
import json
import time
from rdflib import ConjunctiveGraph, URIRef, Literal
from rdflib.namespace import RDF, RDFS
from langdetect import detect
from langdetect.lang_detect_exception import LangDetectException

#word filtering statistics
total_datasets = 0
skipped_no_keywords = 0
skipped_no_domains = 0
datasets_with_domains = 0

keywords_kept_en = 0
keywords_kept_detected_en = 0

keywords_removed_non_english_tag = 0
keywords_removed_non_english_detected = 0
keywords_removed_detection_error = 0

# INPUT / OUTPUT

INPUT_FOLDER = r"D:\Post Doc Research\european open data portal dump\data_europa_eu_dump\data-gov-be"
OUTPUT_FILE = "gov_data_be.json"

BATCH_SIZE = 25
MAX_FILES = None

# RDF URIs

DCAT_DATASET = URIRef(
    "http://www.w3.org/ns/dcat#Dataset"
)

DCT_TITLE = URIRef(
    "http://purl.org/dc/terms/title"
)

DCT_DESCRIPTION = URIRef(
    "http://purl.org/dc/terms/description"
)

DCAT_KEYWORD = URIRef(
    "http://www.w3.org/ns/dcat#keyword"
)

DCAT_THEME = URIRef(
    "http://www.w3.org/ns/dcat#theme"
)

SKOS_PREFLABEL = URIRef(
    "http://www.w3.org/2004/02/skos/core#prefLabel"
)

# OFFICIAL EU DATA THEME TAXONOMY

EU_DATA_THEME_PREFIX = (
    "http://publications.europa.eu/"
    "resource/authority/data-theme/"
)

# GET FILE LIST

trig_files = [
    os.path.join(INPUT_FOLDER, f)
    for f in os.listdir(INPUT_FOLDER)
    if f.endswith(".trig")
]

trig_files = sorted(trig_files)

if MAX_FILES:
    trig_files = trig_files[:MAX_FILES]

print(f"Processing {len(trig_files)} files")
print(f"Batch size: {BATCH_SIZE}\n")

# INIT OUTPUT

f_out = open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
)

f_out.write("[\n")

first = True

total_datasets = 0
skipped_no_keywords = 0
skipped_no_domains = 0
datasets_with_domains = 0

def is_english_text(text, language):
    """
    Keep only English metadata.

    Accept:
        - Explicitly English (@en)
        - Untagged text detected as English

    Reject:
        - Explicitly non-English text
        - Untagged text detected as non-English
        - Detection failures
    """

    text = text.strip()

    if not text:
        return False

    if language == "en":

        if len(text.split()) <= 2:
            return True

        try:
            return detect(text) == "en"

        except LangDetectException:
            return False

    if language is not None:
        return False

    try:
        return detect(text) == "en"

    except LangDetectException:
        return False
        
def keep_keyword(keyword, language):
    """
    Strict English-only keyword filtering.

    Keep:
        - Explicitly English keywords (@en)
          that also appear to be English
        - Untagged keywords detected as English

    Remove:
        - Explicitly non-English keywords
        - Untagged keywords detected as non-English
        - Keywords whose language cannot be determined
    """

    global keywords_kept_en
    global keywords_kept_detected_en

    global keywords_removed_non_english_tag
    global keywords_removed_non_english_detected
    global keywords_removed_detection_error

    keyword = keyword.strip()

    if not keyword:
        return False

    # Explicitly English

    if language == "en":

        # Very short keywords are unreliable for
        # language detection (e.g. OCR, HIV, GDP)

        if len(keyword.split()) <= 2:

            keywords_kept_en += 1

            return True

        try:

            detected_language = detect(
                keyword
            )

            if detected_language == "en":

                keywords_kept_en += 1

                return True

            keywords_removed_non_english_detected += 1

            return False

        except LangDetectException:

            keywords_removed_detection_error += 1

            return False

    # Explicitly non-English

    if language is not None:

        keywords_removed_non_english_tag += 1

        return False

    # Untagged keyword

    try:

        detected_language = detect(
            keyword
        )

        if detected_language == "en":

            keywords_kept_detected_en += 1

            return True

        keywords_removed_non_english_detected += 1

        return False

    except LangDetectException:

        keywords_removed_detection_error += 1

        return False
        
# PROCESS IN BATCHES

for i in range(0, len(trig_files), BATCH_SIZE):

    batch = trig_files[i:i+BATCH_SIZE]

    print(
        f"\n[LOAD BATCH] "
        f"Files {i+1} → {i+len(batch)}"
    )

    g = ConjunctiveGraph()

    batch_start = time.time()

    # LOAD FILES

    for filepath in batch:

        filename = os.path.basename(filepath)

        print(f"[LOAD ] {filename}")

        try:
            g.parse(filepath, format="trig")

        except Exception as e:
            print(f"[ERROR] {filename}: {e}")

    print(
        f"[INFO ] Batch loaded in "
        f"{time.time() - batch_start:.2f}s"
    )

    # FIND DATASETS

    extract_start = time.time()

    datasets = set(
        s for s, _, _ in g.triples(
            (None, RDF.type, DCAT_DATASET)
        )
    )

    print(
        f"[INFO ] Found "
        f"{len(datasets)} datasets"
    )

    dataset_data = {}

    # SINGLE PASS EXTRACTION

    for s, p, o in g:

        if s not in datasets:
            continue

        if s not in dataset_data:

            dataset_data[s] = {
                "title": [],
                "description": [],
                "keywords": set(),
                "domains": set()
            }

       
        # TITLE
        # English + untagged
        if (
            p == DCT_TITLE
            and isinstance(o, Literal)
        ):

            clean_title = str(o).strip()

            if is_english_text(
                clean_title,
                o.language
            ):

                dataset_data[s]["title"].append(
                    clean_title
                )

        # DESCRIPTION

        elif (
            p == DCT_DESCRIPTION
            and isinstance(o, Literal)
        ):

            clean_description = str(o).strip()

            if is_english_text(
                clean_description,
                o.language
            ):

                dataset_data[s]["description"].append(
                    clean_description
                )
                # KEYWORDS
        # EXACT portal keywords

        elif (
            p == DCAT_KEYWORD
            and isinstance(o, Literal)
        ):

            keyword = str(o).strip()

            if keep_keyword(
                keyword,
                o.language
            ):

                dataset_data[s]["keywords"].add(
                    keyword
                )

        # DOMAINS / CATEGORIES
        # EXACT EU portal categories

        elif p == DCAT_THEME:

            theme_uri = str(o)

            # Keep ONLY official EU data themes
            if not theme_uri.startswith(
                EU_DATA_THEME_PREFIX
            ):
                continue

            preferred_label = None

            # CANONICAL PREFLABEL

            for _, _, label in g.triples(
                (o, SKOS_PREFLABEL, None)
            ):

                if (
                    isinstance(label, Literal)
                    and label.language == "en"
                ):

                    preferred_label = str(
                        label
                    ).strip()

                    break

            # FALLBACK RDFS LABEL

            if not preferred_label:

                for _, _, label in g.triples(
                    (o, RDFS.label, None)
                ):

                    if (
                        isinstance(label, Literal)
                        and label.language == "en"
                    ):

                        preferred_label = str(
                            label
                        ).strip()

                        break

            # STORE CATEGOR

            if preferred_label:

                dataset_data[s]["domains"].add(
                    preferred_label
                )

    # WRITE RESULTS

    batch_count = 0

    for ds, data in dataset_data.items():

        # STRICT FILTER:
        # title + keywords + domains

        # STRICT FILTER:
# title + description + keywords + domains

        if not data["title"]:
            continue

        if not data["description"]:
            continue

        if not data["keywords"]:
            skipped_no_keywords += 1
            continue

        if not data["domains"]:
            skipped_no_domains += 1
            continue
        # FINAL SAFETY CHECK

        if not is_english_text(
            data["title"][0],
             "en"
):
            continue

        if not is_english_text(
            data["description"][0],
            "en"
):
            continue
        datasets_with_domains += 1

        record = {

            "id": str(ds),

            "title": data["title"][0],

            "description": data["description"][0],

            "keywords": sorted(
                list(data["keywords"])
            ),

            "domains": sorted(
                list(data["domains"])
            )
        }

        if not first:
            f_out.write(",\n")

        f_out.write(
            json.dumps(
                record,
                ensure_ascii=False
            )
        )

        first = False

        batch_count += 1
        total_datasets += 1

    print(
        f"[DONE ] Batch processed: "
        f"{batch_count} datasets "
        f"({time.time() - extract_start:.2f}s)"
    )

    print(
        f"[INFO ] With domains: "
        f"{datasets_with_domains}"
    )

    print(
        f"[INFO ] Skipped (no keywords): "
        f"{skipped_no_keywords}"
    )

    print(
        f"[INFO ] Skipped (no domains): "
        f"{skipped_no_domains}"
    )

print("\nKeyword Language Filtering")

print(
    f"Kept (@en): "
    f"{keywords_kept_en}"
)

print(
    f"Kept (detected English): "
    f"{keywords_kept_detected_en}"
)

print(
    f"Removed (non-English tag): "
    f"{keywords_removed_non_english_tag}"
)

print(
    f"Removed (detected non-English): "
    f"{keywords_removed_non_english_detected}"
)

print(
    f"Removed (detection error): "
    f"{keywords_removed_detection_error}"
)

    # Free memory
del g

# CLOSE FILE

f_out.write("\n]\n")
f_out.close()

# FINAL STATS

print("\nDone.")

print(
    f"Extracted datasets: "
    f"{total_datasets}"
)

print(
    f"Datasets with domains: "
    f"{datasets_with_domains}"
)

print(
    f"Skipped (no keywords): "
    f"{skipped_no_keywords}"
)

print(
    f"Skipped (no domains): "
    f"{skipped_no_domains}"
)

Processing 107 files
Batch size: 25


[LOAD BATCH] Files 1 → 25
[LOAD ] Dataset_04be205b-8345-4571-ba08-1c43567d8e89.trig


C:\Users\deper\AppData\Local\Temp\ipykernel_10728\1094416696.py:240: DeprecationWarning: ConjunctiveGraph is deprecated, use Dataset instead.
  g = ConjunctiveGraph()


[LOAD ] Dataset_059ea7b6-2ce0-4cbd-9e15-5a5d868950e5.trig
[LOAD ] Dataset_0c5eda3e-c42d-4f5f-99f3-1ef170f5171c.trig
[LOAD ] Dataset_176a0883-432a-43c9-9a11-263ce6f0a340.trig
[LOAD ] Dataset_186b1e68-9d32-4623-b532-e139323991c1.trig
[LOAD ] Dataset_19eb165d-e1fe-48b4-9699-7f92d6692550.trig
[LOAD ] Dataset_1c4a62dc-fb7b-4180-b14c-eadbb34b7d6e.trig
[LOAD ] Dataset_1cd2b12d-0acb-490f-bfdc-482372a2db99.trig
[LOAD ] Dataset_1eefcbd1-f249-4c25-a93b-27c568de445c.trig
[LOAD ] Dataset_1f8370d1-6683-49c7-92d1-d3dac963fe87.trig
[LOAD ] Dataset_220e8943-9ed6-41f5-bca5-9745544b3fc7.trig
[LOAD ] Dataset_251939ac-307a-4a51-b77e-64dd0fae7d8c.trig
[LOAD ] Dataset_28041562-46aa-41f4-b3f4-5718ad823cb2.trig
[LOAD ] Dataset_28a52d59-30d4-4f66-bd4b-59922e6bb914.trig
[LOAD ] Dataset_293067fb-708d-4557-a937-808905f3ade7.trig
[LOAD ] Dataset_2a574aa9-5ab8-4932-bd1e-a4a1599b8dc2.trig
[LOAD ] Dataset_2e4da9f2-dfd0-4199-8175-f9ff99d6b25f.trig
[LOAD ] Dataset_2f00365c-e50e-4a33-80c6-97c64f7a67e3.trig
[LOAD ] Datase